# MOL_LAB DIDACTIC v0.1

**Molekülwerkstatt für den Chemieunterricht der SEK2**

Ziel: Von der Molekülstruktur zu Stoffeigenschaften schließen.

Didaktischer Ablauf: **Beobachten → Vermuten → Antwort prüfen → Erklärung anzeigen**.

Diese erste Version nutzt bewusst nur schnelle, robuste Open-Source-Bausteine: RDKit, py3Dmol, ipywidgets und pandas. xTB/CREST bleiben im großen MOL_LAB.


## 1. Installation

In Google Colab: diese Zelle einmal ausführen. Die Installation dauert normalerweise unter einer Minute.


In [ ]:
!pip -q install rdkit py3Dmol ipywidgets pandas

## 2. Datenbank und Funktionen laden

Diese Zelle enthält die Beispiele, didaktischen Karten, Funktionsgruppen-Erkennung, 2D/3D-Darstellung und die Oberfläche.


In [ ]:

import json, math, textwrap
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
from rdkit import Chem
from rdkit.Chem import AllChem, Draw, Descriptors, rdMolDescriptors, Crippen
from rdkit.Chem.Draw import IPythonConsole
import py3Dmol

MOLECULES = {"Polarität und Löslichkeit": [{"name": "Wasser", "smiles": "O", "niveau": "Basis", "leitfrage": "Warum ist Wasser ein stark polares Lösungsmittel?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Wasser kann Wasserstoffbrücken spenden und aufnehmen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Wasser kann Wasserstoffbrücken spenden und aufnehmen.", "erklaerung": "Wasser besitzt polare O-H-Bindungen und eine gewinkelte Struktur; dadurch heben sich die Dipole nicht auf.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Methan", "smiles": "C", "niveau": "Basis", "leitfrage": "Warum ist Methan unpolar und schlecht wasserlöslich?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Methan ist weitgehend unpolar und kann keine Wasserstoffbrücken bilden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Methan ist weitgehend unpolar und kann keine Wasserstoffbrücken bilden.", "erklaerung": "Methan ist sehr symmetrisch und besitzt keine stark polare funktionelle Gruppe.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ammoniak", "smiles": "N", "niveau": "Basis", "leitfrage": "Warum ist Ammoniak polar und gut wasserlöslich?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ammoniak kann Wasserstoffbrücken bilden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ammoniak kann Wasserstoffbrücken bilden.", "erklaerung": "Ammoniak besitzt polare N-H-Bindungen und ein freies Elektronenpaar am Stickstoff.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Kohlenstoffdioxid", "smiles": "O=C=O", "niveau": "Basis", "leitfrage": "Warum ist CO₂ trotz polarer C=O-Bindungen insgesamt kaum polar?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die lineare symmetrische Struktur hebt die Bindungsdipole auf.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die lineare symmetrische Struktur hebt die Bindungsdipole auf.", "erklaerung": "CO₂ besitzt polare Bindungen, aber die lineare Anordnung führt zur Aufhebung der Dipole.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Methanol", "smiles": "CO", "niveau": "Basis", "leitfrage": "Warum ist Methanol sehr gut mit Wasser mischbar?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die OH-Gruppe kann Wasserstoffbrücken bilden und der unpolare Rest ist klein.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die OH-Gruppe kann Wasserstoffbrücken bilden und der unpolare Rest ist klein.", "erklaerung": "Methanol besitzt eine polare OH-Gruppe und nur eine kleine Methylgruppe.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ethanol", "smiles": "CCO", "niveau": "Basis", "leitfrage": "Warum ist Ethanol mit Wasser mischbar, obwohl es auch einen unpolaren Teil besitzt?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die OH-Gruppe kann Wasserstoffbrücken mit Wasser bilden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die OH-Gruppe kann Wasserstoffbrücken mit Wasser bilden.", "erklaerung": "Ethanol besitzt eine polare OH-Gruppe und einen noch kleinen unpolaren Ethylrest.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "n-Butanol", "smiles": "CCCCO", "niveau": "Basis", "leitfrage": "Warum ist n-Butanol schlechter wasserlöslich als Methanol oder Ethanol?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Der größere unpolare Kohlenwasserstoffrest verringert die Wasserlöslichkeit.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Der größere unpolare Kohlenwasserstoffrest verringert die Wasserlöslichkeit.", "erklaerung": "Die OH-Gruppe bleibt polar, aber mit wachsender Alkylkette nimmt der unpolare Anteil zu.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Methanal", "smiles": "C=O", "niveau": "Basis", "leitfrage": "Warum ist Methanal polar, obwohl es keine OH-Gruppe besitzt?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die Carbonylgruppe ist polar und kann H-Brücken aufnehmen, aber nicht spenden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die Carbonylgruppe ist polar und kann H-Brücken aufnehmen, aber nicht spenden.", "erklaerung": "Die C=O-Doppelbindung ist polar; der Sauerstoff kann als Akzeptor wirken.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Aceton", "smiles": "CC(=O)C", "niveau": "Basis", "leitfrage": "Warum ist Aceton gut mit Wasser mischbar, obwohl es keine OH-Gruppe besitzt?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die C=O-Gruppe kann H-Brücken von Wasser aufnehmen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die C=O-Gruppe kann H-Brücken von Wasser aufnehmen.", "erklaerung": "Aceton besitzt eine polare Carbonylgruppe und zwei kleine unpolare Methylgruppen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Diethylether", "smiles": "CCOCC", "niveau": "Basis", "leitfrage": "Warum unterscheidet sich Diethylether deutlich von Alkoholen?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ether können H-Brücken aufnehmen, aber nicht spenden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ether können H-Brücken aufnehmen, aber nicht spenden.", "erklaerung": "Diethylether enthält ein O-Atom, aber keine O-H-Bindung.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Essigsäure", "smiles": "CC(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Essigsäure stark polar und gut wasserlöslich?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die Carboxylgruppe ist polar und kann H-Brücken bilden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die Carboxylgruppe ist polar und kann H-Brücken bilden.", "erklaerung": "Essigsäure besitzt eine Carboxylgruppe mit C=O und O-H am selben Kohlenstoffatom.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}], "Isomerie": [{"name": "Ethanol", "smiles": "CCO", "niveau": "Basis", "leitfrage": "Warum hat Ethanol andere Eigenschaften als Dimethylether, obwohl beide C₂H₆O besitzen?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ethanol besitzt eine OH-Gruppe und kann H-Brücken spenden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ethanol besitzt eine OH-Gruppe und kann H-Brücken spenden.", "erklaerung": "Gleiche Summenformel, aber andere Verknüpfung der Atome führt zu anderen funktionellen Gruppen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Dimethylether", "smiles": "COC", "niveau": "Basis", "leitfrage": "Warum siedet Dimethylether viel niedriger als Ethanol?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Dimethylether kann keine Wasserstoffbrücken spenden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Dimethylether kann keine Wasserstoffbrücken spenden.", "erklaerung": "Er besitzt eine Ethergruppe statt einer Alkoholgruppe.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Propanal", "smiles": "CCC=O", "niveau": "Basis", "leitfrage": "Wie unterscheidet sich Propanal von Aceton?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Propanal ist ein Aldehyd.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Propanal ist ein Aldehyd.", "erklaerung": "Die Carbonylgruppe liegt am Ende der Kette.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Aceton", "smiles": "CC(=O)C", "niveau": "Basis", "leitfrage": "Wie unterscheidet sich Aceton von Propanal?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Aceton ist ein Keton.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Aceton ist ein Keton.", "erklaerung": "Die Carbonylgruppe liegt zwischen zwei Kohlenstoffatomen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Butan", "smiles": "CCCC", "niveau": "Basis", "leitfrage": "Was unterscheidet Butan von Isobutan?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Butan ist unverzweigt, Isobutan ist verzweigt.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Butan ist unverzweigt, Isobutan ist verzweigt.", "erklaerung": "Beide haben dieselbe Summenformel, aber unterschiedliche Kohlenstoffverknüpfung.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Isobutan", "smiles": "CC(C)C", "niveau": "Basis", "leitfrage": "Warum ist Isobutan kompakter als Butan?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die Kohlenstoffkette ist verzweigt.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die Kohlenstoffkette ist verzweigt.", "erklaerung": "Verzweigung verändert Molekülform und damit Stoffeigenschaften.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Propansäure", "smiles": "CCC(=O)O", "niveau": "Basis", "leitfrage": "Warum unterscheidet sich Propansäure stark von Essigsäuremethylester?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Propansäure enthält eine Carboxylgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Propansäure enthält eine Carboxylgruppe.", "erklaerung": "Funktionelle Isomerie: Carbonsäure vs. Ester.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Essigsäuremethylester", "smiles": "CC(=O)OC", "niveau": "Basis", "leitfrage": "Warum ist Essigsäuremethylester kein Säure-Isomer im Eigenschaftssinn?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Er enthält eine Estergruppe und keine Carboxyl-OH-Gruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Er enthält eine Estergruppe und keine Carboxyl-OH-Gruppe.", "erklaerung": "Gleiche Summenformel kann durch andere funktionelle Gruppe ganz andere Eigenschaften ergeben.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "D-Alanin", "smiles": "C[C@@H](N)C(=O)O", "niveau": "Basis", "leitfrage": "Warum sind D- und L-Alanin trotz gleicher Verknüpfung nicht dasselbe Molekül?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Sie sind Enantiomere, also spiegelbildliche Moleküle.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Sie sind Enantiomere, also spiegelbildliche Moleküle.", "erklaerung": "Chiralität zeigt, dass auch die räumliche Anordnung entscheidend ist.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "L-Alanin", "smiles": "C[C@H](N)C(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Chiralität bei Aminosäuren biologisch wichtig?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["L-Alanin ist eine spiegelbildliche Form von D-Alanin.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: L-Alanin ist eine spiegelbildliche Form von D-Alanin.", "erklaerung": "Biologische Systeme unterscheiden Spiegelbilder häufig sehr genau.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Glucose offen", "smiles": "O=CC(O)C(O)C(O)C(O)CO", "niveau": "Basis", "leitfrage": "Warum ist Glucose ein komplexeres Isomeriebeispiel?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Offenkettige Glucose ist eine Aldose.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Offenkettige Glucose ist eine Aldose.", "erklaerung": "Glucose und Fructose haben dieselbe Summenformel, unterscheiden sich aber in der Carbonylgruppe.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Fructose offen", "smiles": "OCC(O)C(O)C(O)C(=O)CO", "niveau": "Basis", "leitfrage": "Warum ist Fructose trotz gleicher Summenformel anders als Glucose?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Fructose ist eine Ketose.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Fructose ist eine Ketose.", "erklaerung": "In Lösung sind zusätzlich Ringformen wichtig; hier wird bewusst vereinfacht.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}], "Funktionelle Gruppen": [{"name": "Methanol", "smiles": "CO", "niveau": "Basis", "leitfrage": "Woran erkennt man einen Alkohol?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Methanol besitzt eine Alkoholgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Methanol besitzt eine Alkoholgruppe.", "erklaerung": "Eine OH-Gruppe an einem gesättigten Kohlenstoff prägt Polarität und H-Brücken.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ethanal", "smiles": "CC=O", "niveau": "Basis", "leitfrage": "Woran erkennt man einen Aldehyd?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ethanal besitzt eine Aldehydgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ethanal besitzt eine Aldehydgruppe.", "erklaerung": "Die Carbonylgruppe liegt am Kettenende.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Aceton", "smiles": "CC(=O)C", "niveau": "Basis", "leitfrage": "Woran erkennt man ein Keton?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Aceton besitzt eine Ketogruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Aceton besitzt eine Ketogruppe.", "erklaerung": "Die Carbonylgruppe liegt zwischen zwei Kohlenstoffresten.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Essigsäure", "smiles": "CC(=O)O", "niveau": "Basis", "leitfrage": "Woran erkennt man eine Carbonsäure?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Essigsäure besitzt eine Carboxylgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Essigsäure besitzt eine Carboxylgruppe.", "erklaerung": "C=O und OH am selben Kohlenstoffatom bilden die Carboxylgruppe.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ethylacetat", "smiles": "CC(=O)OCC", "niveau": "Basis", "leitfrage": "Woran erkennt man einen Ester?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ethylacetat besitzt eine Estergruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ethylacetat besitzt eine Estergruppe.", "erklaerung": "Das Strukturmotiv ist C(=O)-O-C.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Diethylether", "smiles": "CCOCC", "niveau": "Basis", "leitfrage": "Woran erkennt man einen Ether?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Diethylether besitzt eine Ethergruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Diethylether besitzt eine Ethergruppe.", "erklaerung": "Ein Sauerstoffatom ist mit zwei Kohlenstoffresten verbunden.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Acetanhydrid", "smiles": "CC(=O)OC(=O)C", "niveau": "Basis", "leitfrage": "Woran erkennt man ein Carbonsäureanhydrid?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Acetanhydrid besitzt eine Anhydridgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Acetanhydrid besitzt eine Anhydridgruppe.", "erklaerung": "Zwei Acylgruppen sind über ein Sauerstoffatom verbunden.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ethylamin", "smiles": "CCN", "niveau": "Basis", "leitfrage": "Woran erkennt man ein Amin?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ethylamin besitzt eine Aminogruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ethylamin besitzt eine Aminogruppe.", "erklaerung": "Amine enthalten Stickstoff und können basische Eigenschaften zeigen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Acetamid", "smiles": "CC(=O)N", "niveau": "Basis", "leitfrage": "Woran erkennt man ein Amid?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Acetamid besitzt eine Amidgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Acetamid besitzt eine Amidgruppe.", "erklaerung": "C(=O)-N ist das zentrale Motiv; die Peptidbindung ist ebenfalls ein Amid.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Phenol", "smiles": "c1ccccc1O", "niveau": "Basis", "leitfrage": "Warum ist Phenol nicht einfach ein normaler Alkohol?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die OH-Gruppe ist direkt an einen aromatischen Ring gebunden.", "erklaerung": "Phenole unterscheiden sich in Eigenschaften und Reaktivität von einfachen Alkoholen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Anilin", "smiles": "c1ccccc1N", "niveau": "Basis", "leitfrage": "Warum ist Anilin ein aromatisches Amin?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Anilin besitzt eine Aminogruppe am aromatischen Ring.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Anilin besitzt eine Aminogruppe am aromatischen Ring.", "erklaerung": "Aromat und Aminogruppe beeinflussen sich gegenseitig.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}], "Konformere": [{"name": "Butan", "smiles": "CCCC", "niveau": "Basis", "leitfrage": "Warum gibt es bei Butan mehrere räumliche Formen?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Konformere entstehen durch Rotation um Einfachbindungen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Konformere entstehen durch Rotation um Einfachbindungen.", "erklaerung": "Butan kann um C-C-Einfachbindungen rotieren; verschiedene Formen haben verschiedene relative Energien.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "1,2-Dichlorethan", "smiles": "ClCCCl", "niveau": "Basis", "leitfrage": "Wie beeinflusst Rotation die Orientierung der Chloratome?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Gruppen können im Raum unterschiedlich nahe kommen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Gruppen können im Raum unterschiedlich nahe kommen.", "erklaerung": "Rotation verändert Abstände und Wechselwirkungen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ethylenglykol", "smiles": "OCCO", "niveau": "Basis", "leitfrage": "Warum kann Ethylenglykol mehrere günstige Formen einnehmen?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Mehrere Einfachbindungen können rotieren.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Mehrere Einfachbindungen können rotieren.", "erklaerung": "Die beiden OH-Gruppen und flexible Bindungen ermöglichen mehrere Konformere.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Cyclohexan", "smiles": "C1CCCCC1", "niveau": "Basis", "leitfrage": "Warum ist die Sesselstruktur von Cyclohexan wichtig?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Eine nicht-planare Form verringert ungünstige Spannungen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Eine nicht-planare Form verringert ungünstige Spannungen.", "erklaerung": "Cyclohexan ist nicht planar; die Sesselstruktur ist energetisch günstig.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Milchsäure", "smiles": "C[C@H](O)C(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Milchsäure ein gutes Beispiel für Flexibilität und Chiralität?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Sie besitzt ein chirales Zentrum und mehrere polare Gruppen.", "erklaerung": "Milchsäure verbindet funktionelle Gruppen, Chiralität und Beweglichkeit.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}], "Wirkstoffe und Biomoleküle": [{"name": "Coffein", "smiles": "Cn1cnc2n(C)c(=O)n(C)c(=O)c12", "niveau": "Basis", "leitfrage": "Warum kann Coffein mit Biomolekülen wechselwirken?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Coffein besitzt mehrere H-Brücken-Akzeptoren, aber keine Donoren.", "erklaerung": "Das heteroatomreiche Ringsystem erlaubt mehrere schwache Wechselwirkungen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Aspirin", "smiles": "CC(=O)Oc1ccccc1C(=O)O", "niveau": "Basis", "leitfrage": "Welche funktionellen Gruppen prägen Aspirin?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Aspirin enthält Carboxylgruppe, Estergruppe und Aromat.", "erklaerung": "Polare und unpolare Bereiche kommen in einem Wirkstoffmolekül zusammen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Paracetamol", "smiles": "CC(=O)Nc1ccc(O)cc1", "niveau": "Basis", "leitfrage": "Welche Strukturmerkmale machen Paracetamol polar?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Paracetamol enthält Phenol und Amid.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Paracetamol enthält Phenol und Amid.", "erklaerung": "Diese Gruppen können H-Brücken bilden; der Aromat ist unpolarer.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ibuprofen", "smiles": "CC(C)Cc1ccc(cc1)[C@@H](C)C(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Ibuprofen relativ lipophil?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Es besitzt eine polare Carboxylgruppe und einen großen unpolaren Bereich.", "erklaerung": "Wirkstoffe enthalten oft beide Bereiche: polar für Wechselwirkungen, unpolar für Bindungstaschen/Membranen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Acetazolamid", "smiles": "CC(=O)Nc1nnc(s1)S(N)(=O)=O", "niveau": "Basis", "leitfrage": "Warum ist Acetazolamid ein Brückenbeispiel zum Proteinlabor?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Es besitzt mehrere polare Gruppen und Heteroatome.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Es besitzt mehrere polare Gruppen und Heteroatome.", "erklaerung": "Als Carboanhydrase-Hemmstoff eignet es sich für Struktur-Wirkungs- und Protein-Ligand-Bezüge.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Glycin", "smiles": "NCC(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Glycin die einfachste Aminosäure?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Glycin besitzt Aminogruppe und Carboxylgruppe.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Glycin besitzt Aminogruppe und Carboxylgruppe.", "erklaerung": "Die Seitenkette ist nur ein Wasserstoffatom.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Alanin", "smiles": "C[C@H](N)C(=O)O", "niveau": "Basis", "leitfrage": "Warum ist Alanin chiral?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Das zentrale C-Atom trägt vier verschiedene Gruppen.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Das zentrale C-Atom trägt vier verschiedene Gruppen.", "erklaerung": "Alanin ist ein einfaches Beispiel für Chiralität in Biomolekülen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Phenylalanin", "smiles": "N[C@@H](Cc1ccccc1)C(=O)O", "niveau": "Basis", "leitfrage": "Was unterscheidet Phenylalanin von Glycin und Alanin?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Die Seitenkette enthält einen aromatischen Ring.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Die Seitenkette enthält einen aromatischen Ring.", "erklaerung": "Aromatische hydrophobe Seitenketten sind wichtig für Proteinstruktur und Wechselwirkungen.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Adenin", "smiles": "Nc1ncnc2ncnc12", "niveau": "Basis", "leitfrage": "Warum ist Adenin als Nukleobase interessant?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Adenin ist eine stickstoffhaltige Nukleobase.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Adenin ist eine stickstoffhaltige Nukleobase.", "erklaerung": "Das heteroaromatische Ringsystem kann an Basenpaarung beteiligt sein.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}, {"name": "Ribose", "smiles": "O=CC(O)C(O)C(O)CO", "niveau": "Basis", "leitfrage": "Warum ist Ribose ein wichtiger RNA-Baustein?", "beobachten": ["Betrachte zuerst die 2D-Struktur.", "Drehe anschließend das 3D-Modell.", "Vergleiche die Deskriptoren mit der Leitfrage."], "frage": "Welche Aussage passt am besten?", "antworten": ["Ribose besitzt mehrere OH-Gruppen und ist stark polar.", "Die Summenformel allein erklärt alle Stoffeigenschaften.", "Das Molekül ist automatisch ionisch, wenn Sauerstoff oder Stickstoff vorkommt.", "Die räumliche Struktur spielt für Eigenschaften keine Rolle."], "richtig": 0, "feedback": "Richtig: Ribose besitzt mehrere OH-Gruppen und ist stark polar.", "erklaerung": "In RNA ist Ribose Teil des Zucker-Phosphat-Rückgrats; hier wird die offenkettige Form vereinfacht gezeigt.", "modellgrenze": "Die App liefert Modellwerte und Strukturhinweise. Experimente und Kontext können dadurch nicht vollständig ersetzt werden."}]}

FUNCTIONAL_GROUPS = {
    "Alkohol": "[OX2H][CX4]",
    "Phenol": "[OX2H]c",
    "Aldehyd": "[CX3H1](=O)[#6,#1]",
    "Keton": "[#6][CX3](=O)[#6]",
    "Carbonsäure": "C(=O)[OX2H1]",
    "Ester": "C(=O)O[#6]",
    "Ether": "[#6][OD2][#6]",
    "Carbonsäureanhydrid": "C(=O)OC(=O)",
    "Amin": "[NX3;H2,H1,H0;!$(NC=O)]",
    "Amid": "C(=O)N",
    "Aromatischer Ring": "a",
    "Carbonylgruppe": "[CX3]=[OX1]",
}

current_report = {}

def mol_from_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError("Ungültiger SMILES-Code.")
    return mol

def make_3d(mol):
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3()
    params.randomSeed = 42
    status = AllChem.EmbedMolecule(mol_h, params)
    if status != 0:
        AllChem.EmbedMolecule(mol_h, randomSeed=42)
    try:
        AllChem.MMFFOptimizeMolecule(mol_h, maxIters=300)
    except Exception:
        try:
            AllChem.UFFOptimizeMolecule(mol_h, maxIters=300)
        except Exception:
            pass
    return mol_h

def calc_properties(mol):
    return {
        "Summenformel": rdMolDescriptors.CalcMolFormula(mol),
        "Molmasse / g mol⁻¹": round(Descriptors.MolWt(mol), 2),
        "H-Brücken-Donoren": rdMolDescriptors.CalcNumHBD(mol),
        "H-Brücken-Akzeptoren": rdMolDescriptors.CalcNumHBA(mol),
        "logP (Modellwert)": round(Crippen.MolLogP(mol), 2),
        "TPSA / Å²": round(rdMolDescriptors.CalcTPSA(mol), 2),
        "rotierbare Bindungen": rdMolDescriptors.CalcNumRotatableBonds(mol),
        "Ringanzahl": rdMolDescriptors.CalcNumRings(mol),
        "schwere Atome": mol.GetNumHeavyAtoms(),
    }

def detect_functional_groups(mol):
    found=[]
    for name, smarts in FUNCTIONAL_GROUPS.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt is not None and mol.HasSubstructMatch(patt):
            found.append(name)
    return sorted(set(found))

def show_2d(mol, width=360, height=260):
    img = Draw.MolToImage(mol, size=(width, height))
    display(img)

def show_3d(mol3d, width=460, height=360):
    mb = Chem.MolToMolBlock(mol3d)
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(mb, "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    viewer.show()

def interpret_properties(props):
    parts=[]
    logp=props.get("logP (Modellwert)",0)
    tpsa=props.get("TPSA / Å²",0)
    hbd=props.get("H-Brücken-Donoren",0)
    hba=props.get("H-Brücken-Akzeptoren",0)
    rot=props.get("rotierbare Bindungen",0)
    if hbd>0: parts.append("Das Molekül kann Wasserstoffbrücken spenden.")
    else: parts.append("Das Molekül kann keine Wasserstoffbrücken spenden.")
    if hba>0: parts.append("Es kann Wasserstoffbrücken aufnehmen.")
    if tpsa < 20: parts.append("Die berechnete polare Oberfläche ist klein; das spricht für eher unpolare Eigenschaften.")
    elif tpsa < 80: parts.append("Die berechnete polare Oberfläche ist mittelgroß; polare und unpolare Bereiche können nebeneinander vorkommen.")
    else: parts.append("Die berechnete polare Oberfläche ist groß; deutliche polare Bereiche sind zu erwarten.")
    if logp < 0: parts.append("Der logP-Wert spricht eher für hydrophile Eigenschaften.")
    elif logp < 3: parts.append("Der logP-Wert spricht für ein gemischtes Verhältnis zwischen hydrophilen und lipophilen Anteilen.")
    else: parts.append("Der logP-Wert spricht für ein eher lipophiles Molekül.")
    if rot >= 5: parts.append("Mehrere rotierbare Bindungen machen das Molekül vergleichsweise flexibel.")
    elif rot >= 1: parts.append("Das Molekül besitzt einige bewegliche Einfachbindungen.")
    else: parts.append("Das Molekül ist relativ starr.")
    return " ".join(parts)

def generate_conformers(smiles, n=20):
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    params = AllChem.ETKDGv3(); params.randomSeed = 42
    ids = list(AllChem.EmbedMultipleConfs(mol, numConfs=n, params=params))
    results=[]
    props = AllChem.MMFFGetMoleculeProperties(mol)
    for cid in ids:
        try:
            ff = AllChem.MMFFGetMoleculeForceField(mol, props, confId=cid)
            ff.Minimize(maxIts=300)
            e = ff.CalcEnergy()
        except Exception:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff.Minimize(maxIts=300)
            e = ff.CalcEnergy()
        results.append((cid,e))
    if not results: return mol, []
    mine=min(e for _,e in results)
    rel=[(cid, round(e-mine,2)) for cid,e in results]
    rel.sort(key=lambda x:x[1])
    return mol, rel

def show_conf(mol, conf_id, title="Konformer"):
    mb = Chem.MolToMolBlock(mol, confId=conf_id)
    viewer = py3Dmol.view(width=340, height=280)
    viewer.addModel(mb, "mol")
    viewer.setStyle({"stick": {}})
    viewer.zoomTo()
    display(HTML(f"<b>{title}</b>"))
    viewer.show()

def card_for(module, name):
    for c in MOLECULES[module]:
        if c["name"] == name:
            return c
    return None

def report_text(rep):
    lines=[]
    lines.append(f"MOL_LAB DIDACTIC – Kurzanalyse")
    lines.append(f"Molekül: {rep.get('name','')}")
    lines.append(f"SMILES: {rep.get('smiles','')}")
    lines.append("")
    lines.append("Eigenschaften:")
    for k,v in rep.get("props",{}).items(): lines.append(f"- {k}: {v}")
    lines.append("")
    lines.append("Funktionelle Gruppen: " + (", ".join(rep.get('groups',[])) or "keine erkannt"))
    lines.append("")
    lines.append("Leitfrage: " + rep.get('leitfrage',''))
    lines.append("Modell-Deutung: " + rep.get('interpretation',''))
    lines.append("Erklärung: " + rep.get('erklaerung',''))
    lines.append("Modellgrenze: " + rep.get('modellgrenze',''))
    return "\n".join(lines)

# Oberfläche
module_dd = widgets.Dropdown(options=list(MOLECULES.keys()), description="Modul:", layout=widgets.Layout(width="520px"))
example_dd = widgets.Dropdown(description="Beispiel:", layout=widgets.Layout(width="520px"))
custom_smiles = widgets.Text(value="", placeholder="optional eigener SMILES-Code, z.B. CCO", description="SMILES:", layout=widgets.Layout(width="700px"))
use_custom = widgets.Checkbox(value=False, description="eigenen SMILES verwenden")
run_btn = widgets.Button(description="Molekül analysieren", button_style="success")
answer_radio = widgets.RadioButtons(options=[], description="Antwort:", layout=widgets.Layout(width="900px"))
check_btn = widgets.Button(description="Antwort prüfen", button_style="info")
explain_btn = widgets.Button(description="Erklärung anzeigen")
report_btn = widgets.Button(description="Analyse als Text anzeigen")
conf_btn = widgets.Button(description="Konformere berechnen", button_style="warning")
out = widgets.Output()
feedback_out = widgets.Output()
report_out = widgets.Output()


def refresh_examples(*args):
    module=module_dd.value
    example_dd.options=[c["name"] for c in MOLECULES[module]]
refresh_examples()
module_dd.observe(refresh_examples, names="value")

def analyze(_=None):
    global current_report
    with out:
        clear_output()
        module=module_dd.value
        if use_custom.value:
            smiles=custom_smiles.value.strip()
            if not smiles:
                print("Bitte einen SMILES-Code eingeben oder das Häkchen entfernen."); return
            card={"name":"Eigenes Molekül", "smiles":smiles, "leitfrage":"Welche Strukturmerkmale könnten die Stoffeigenschaften beeinflussen?", "beobachten":["Suche polare Gruppen.","Achte auf die Molekülgröße.","Vergleiche logP, TPSA und H-Brücken."], "frage":"Welche Aussage ist bei eigenen Molekülen besonders wichtig?", "antworten":["Die berechneten Werte sind Modellwerte und müssen fachlich geprüft werden.","SMILES-Eingaben sind immer experimentelle Messwerte.","Die Summenformel allein reicht immer aus.","Große Moleküle sind in dieser App immer zuverlässig berechnet."], "richtig":0, "feedback":"Richtig: Die App gibt Modellhinweise, keine endgültigen Messwerte.", "erklaerung":"Eigene SMILES-Eingaben sind für kleine bis mittelkleine Moleküle gedacht. Prüfe immer, ob die Struktur chemisch sinnvoll ist.", "modellgrenze":"Für große, geladene oder ungewöhnliche Moleküle ist diese einfache RDKit-Analyse nur eingeschränkt geeignet."}
        else:
            card=card_for(module, example_dd.value)
            smiles=card["smiles"]
        try:
            mol=mol_from_smiles(smiles)
        except Exception as e:
            print("Fehler:", e); return
        props=calc_properties(mol)
        groups=detect_functional_groups(mol)
        interp=interpret_properties(props)
        display(HTML(f"<h2>{card['name']}</h2><p><b>SMILES:</b> <code>{smiles}</code></p>"))
        display(HTML(f"<h3>Leitfrage</h3><p>{card.get('leitfrage','')}</p>"))
        display(HTML("<h3>Beobachte</h3><ul>" + "".join(f"<li>{x}</li>" for x in card.get('beobachten',[])) + "</ul>"))
        display(HTML("<h3>2D-Struktur</h3>")); show_2d(mol)
        display(HTML("<h3>3D-Modell</h3>")); show_3d(make_3d(mol))
        display(HTML("<h3>Eigenschaften</h3>")); display(pd.DataFrame(list(props.items()), columns=["Eigenschaft","Wert"]))
        display(HTML("<h3>Erkannte funktionelle Gruppen</h3><p>" + (", ".join(groups) if groups else "keine aus der einfachen Liste erkannt") + "</p>"))
        display(HTML("<h3>Erste Modell-Deutung</h3><p>" + interp + "</p>"))
        answer_radio.options=card.get("antworten",[])
        answer_radio.value=answer_radio.options[0] if answer_radio.options else None
        display(HTML("<h3>Vermute</h3><p>" + card.get('frage','') + "</p>"))
        display(answer_radio, widgets.HBox([check_btn, explain_btn, report_btn]))
        current_report={"name":card["name"], "smiles":smiles, "card":card, "props":props, "groups":groups, "leitfrage":card.get("leitfrage",""), "interpretation":interp, "erklaerung":card.get("erklaerung",""), "modellgrenze":card.get("modellgrenze","")}
    feedback_out.clear_output(); report_out.clear_output()

def check_answer(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        card=current_report["card"]
        selected=answer_radio.value
        idx=list(answer_radio.options).index(selected)
        if idx == card.get("richtig",0):
            display(HTML("<p style='color:green'><b>Richtig.</b> " + card.get("feedback","") + "</p>"))
        else:
            right=card.get("antworten",[])[card.get("richtig",0)]
            display(HTML("<p style='color:#b00020'><b>Noch nicht ganz.</b> Die beste Antwort ist: <b>" + right + "</b></p><p>" + card.get("feedback","") + "</p>"))

def explain(_=None):
    with feedback_out:
        clear_output()
        if not current_report: return
        display(HTML("<h3>Erklärung</h3><p>" + current_report.get("erklaerung","") + "</p><h4>Grenze des Modells</h4><p>" + current_report.get("modellgrenze","") + "</p>"))

def show_report(_=None):
    with report_out:
        clear_output()
        if not current_report: return
        txt=report_text(current_report)
        display(widgets.Textarea(value=txt, layout=widgets.Layout(width="100%", height="260px")))

def do_conformers(_=None):
    with report_out:
        clear_output()
        if not current_report: return
        smiles=current_report["smiles"]
        mol0 = Chem.MolFromSmiles(smiles)
        if mol0 is None or mol0.GetNumHeavyAtoms() > 35:
            print("Konformer-Suche nur für kleine bis mittelgroße Moleküle empfohlen."); return
        display(HTML("<h3>Konformere: einfache RDKit/MMFF-Suche</h3><p>Relative Energien sind Modellwerte in kcal/mol.</p>"))
        mol, rel = generate_conformers(smiles, n=25)
        display(pd.DataFrame([{"Konformer":i+1,"confId":cid,"relative Energie / kcal mol⁻¹":e} for i,(cid,e) in enumerate(rel[:5])]))
        for i,(cid,e) in enumerate(rel[:3]): show_conf(mol, cid, f"Konformer {i+1}, ΔE = {e} kcal/mol")

run_btn.on_click(analyze)
check_btn.on_click(check_answer)
explain_btn.on_click(explain)
report_btn.on_click(show_report)
conf_btn.on_click(do_conformers)

display(HTML("<h1>MOL_LAB DIDACTIC</h1><p>Wähle ein Modul und ein Beispiel oder gib einen eigenen SMILES-Code ein.</p>"))
display(module_dd, example_dd, use_custom, custom_smiles, widgets.HBox([run_btn, conf_btn]), out, feedback_out, report_out)


## 3. Hinweise für Version 0.2

Vorbereitet, aber noch nicht umgesetzt:

- einklappbare LehrerInnenhinweise
- sauberer HTML/PDF/Docx-Export
- kuratierte experimentelle Vergleichsdaten, z. B. Siedepunkte und Löslichkeitsklassen
- weitere Beispielsets
- getrennte Arbeitskarten für Basis/Mittel/Vertiefung
